# Modelado y aprendizaje automático

Este cuaderno inspecciona el conjunto gold y ejecuta el entrenamiento reproducible definido en `src/models/train_models.py`. La limpieza y construcción del dataset se realizan previamente con los scripts de `src/data/`.

In [2]:
from pathlib import Path
import sys

import pandas as pd

RAIZ = next(
    (
        ruta
        for ruta in (Path.cwd(), *Path.cwd().parents)
        if (ruta / "src" / "models" / "train_models.py").is_file()
    ),
    None,
)
if RAIZ is None:
    raise FileNotFoundError(
        "No se encuentra la raíz del proyecto. Abre el notebook desde el repositorio."
    )

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.models import train_models

RUTA_GOLD = RAIZ / "data" / "gold" / "alquiler_municipios_dataset.csv"
if not RUTA_GOLD.is_file():
    raise FileNotFoundError(
        f"No existe {RUTA_GOLD}. Ejecuta primero los scripts de src/data/."
    )

gold = pd.read_csv(RUTA_GOLD, dtype={"codigo_territorial": "string"})
print(f"Filas: {len(gold):,} | Columnas: {len(gold.columns)}")
display(gold.head())

Filas: 47,541 | Columnas: 8


,codigo_territorial,municipio,anio,periodo,fecha_periodo,contratos,renta_euros_mes,tramo_precios
0,08001,Abrera,2007,gener-desembre,2007-10-01,340,596.06,> 500 i <= 650 euros/mes
1,08001,Abrera,2008,gener-desembre,2008-10-01,230,637.96,> 500 i <= 650 euros/mes
2,08001,Abrera,2009,gener-desembre,2009-10-01,720,625.55,> 500 i <= 650 euros/mes
3,08001,Abrera,2010,gener-desembre,2010-10-01,1290,641.09,> 500 i <= 650 euros/mes
4,08001,Abrera,2011,gener-desembre,2011-10-01,2260,525.69,> 500 i <= 650 euros/mes


## Comprobaciones del conjunto Gold

Se comprueban fechas interpretables y rentas positivas antes del entrenamiento.

In [3]:
gold_revision = gold.copy()
gold_revision["fecha_periodo"] = pd.to_datetime(
    gold_revision["fecha_periodo"], errors="coerce"
)
gold_revision["renta_euros_mes"] = pd.to_numeric(
    gold_revision["renta_euros_mes"], errors="coerce"
)

resumen = pd.Series(
    {
        "filas": len(gold_revision),
        "municipios": gold_revision["codigo_territorial"].nunique(),
        "fechas interpretables": gold_revision["fecha_periodo"].notna().sum(),
        "rentas positivas": gold_revision["renta_euros_mes"].gt(0).sum(),
        "rentas ausentes": gold_revision["renta_euros_mes"].isna().sum(),
    },
    name="valor",
)
display(resumen.to_frame())

,valor
filas,47541
municipios,945
fechas interpretables,47541
rentas positivas,26594
rentas ausentes,20947


## Entrenamiento y evaluación

Se reutiliza `main()` para mantener en un solo lugar la preparación, la separación temporal, la comparación de modelos y el guardado de artefactos.

In [4]:
train_models.main()

RUTA_METRICAS = RAIZ / "reports" / "metricas_modelos.csv"
display(pd.read_csv(RUTA_METRICAS))

\nComparación de modelos:
                      modelo  MAE_euros  RMSE_euros  MAPE_porcentaje  observaciones_prueba
                       Ridge  75.323695  108.384987        10.130101                  6119
        HistGradientBoosting 111.936159  147.236792        16.475995                  6119
Referencia_mediana_municipal 123.532328  149.757765        17.736148                  6119
\nFecha de corte temporal: 2024-04-01
Modelo guardado para el MVP: Ridge


,modelo,MAE_euros,RMSE_euros,MAPE_porcentaje,observaciones_prueba
0,Ridge,75.323695,108.384987,10.130101,6119
1,HistGradientBoosting,111.936159,147.236792,16.475995,6119
2,Referencia_mediana_municipal,123.532328,149.757765,17.736148,6119


El modelo listo para la aplicación se guarda en `models/modelo_alquiler.joblib`; las métricas, predicciones de prueba y gráficas se guardan en `reports/`.